# [실습] 보고서 작성 에이전트 만들기

이전에 배운 ReAct 구조의 에이전트를 활용하여, 보고서를 작성하는 에이전트를 만들어 보겠습니다.   

기존의 툴이 모두 LLM에게 정보를 전달하는 방식이었는데요.   

이번에 추가할 툴은 LLM이 파일 시스템에 접근하고, 정보를 직접 전달하도록 합니다.

# 라이브러리 설정 및 LLM 불러오기

In [ ]:
!pip install beautifulsoup4 langchain==0.3.27 langchain_community==0.3.27 langgraph==0.6.8 setuptools koreanize_matplotlib matplotlib langchain_experimental langchain_google_genai langchain_openai langchain_tavily

In [ ]:
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3)
    llm_nonreasoning = ChatOpenAI(model='gpt-4.1-mini', temperature=0.3)
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, 
                                rate_limiter=rate_limiter)
    llm_gemini_nonreasoning = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.3, 
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

print("필수 모듈 임포트 완료")

4개의 툴을 활용합니다.

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from langchain_experimental.utilities import PythonREPL
from typing import Annotated

repl = PythonREPL()

@tool
def tavily_search(query, max_results=5):
    """Tavily API를 통해 검색 결과를 가져옵니다.
주어진 주제에 맞는 적절한 argument 값을 선정하세요.
query: 검색어
max_results : 검색 결과의 수(최소 1, 최대 10, 별도의 요청이 없으면 5로 고정)"""

    # Raw Content를 포함하는 검색 시스템
    tavily_search = TavilySearch(max_results=max_results, include_raw_content='markdown')

    search_results = tavily_search.invoke(query)['results']

    context =''
    for doc in search_results:
        # 풀 검색이 가능하면 풀 컨텐츠 전달
        if doc.get('raw_content'):
            doc_content = doc.get('raw_content')
        # 없으면 기존 검색
        else:
            doc_content = doc.get('content')

        context += 'TITLE: ' + doc.get('title','N/A') + '\nURL:' + doc.get('url')+ '\nContent:'+ doc_content+'\n\n---\n\n'
    return context

@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you MUST print it out with `print(...)`. This is visible to the user."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    result_str = f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"
    return result_str

간단한 `WebBaseLoader`를 이용한 fetch_web 툴도 구성합니다.

In [ ]:
@tool
def current_date() -> str:
    "현재 날짜를 %y-%m-%d 형식으로 반환합니다."
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

@tool
def fetch_web(url:str) -> str:
    "자세한 내용이 필요한 경우, 주어진 URL에서 전체 웹 페이지 내용을 가져옵니다."
    from langchain_community.document_loaders import WebBaseLoader
    loader = WebBaseLoader(url)
    docs = loader.load()
    return docs

print(current_date.invoke({}))
print(fetch_web.invoke({'url':'https://www.samsungsds.com/kr/gpuaas/gpuaas.html'}))


In [ ]:
tools = [tavily_search, python_repl_tool, current_date, fetch_web]

React Agent를 구성합니다.

In [ ]:
def react_agent(llm, question , tools = [tavily_search]):

    # 툴과 LLM 구성

    tool_list = {x.name: x for x in tools}
    llm_with_tools = llm.bind_tools(tools)

    # 메시지 구성
    messages = [HumanMessage(content=question)]
    print('Query:', question)

    # LLM에 메시지 전달 (분기)
    tool_msg = llm_with_tools.invoke(question)
    messages.append(tool_msg)
    print('## LLM 호출 ##')
    if tool_msg.content:
        print('LLM:', tool_msg.content)

    while tool_msg.tool_calls:
        # 툴 호출이 있을 경우: 툴 실행 후 결과를 전달 (반복)

        for tool_call in tool_msg.tool_calls:
            tool_name = tool_call['name']

            print(f"-- {tool_name} 사용 중 --")
            print(tool_call)


            tool_exec = tool_list[tool_name]

            tool_result = tool_exec.invoke(tool_call)
            messages.append(tool_result)
        tool_msg = llm_with_tools.invoke(messages)
        print('## LLM 호출 ##')
        messages.append(tool_msg)

        if tool_msg.content:
            print('LLM:', tool_msg.content)

    return messages
    # 메시지 전체 보여주기

In [ ]:
question= '삼성SDS의 GPUaaS 서비스의 주요 고객사를 알려줘.'
response = react_agent(llm=llm, question=question, tools= tools)

In [ ]:
question= '삼성SDS의 GPUaaS 서비스의 주요 고객사를 알려줘.'
response = react_agent(llm=llm_nonreasoning, question=question, tools= tools)

# [실습] 마크다운 보고서 작성 툴 만들기   

content와 file_path, mode를 입력으로 받아, 
특정 경로에 마크다운 파일을 생성하거나 수정하는 툴을 제작하세요.   

아래의 포인트를 고려하세요.

In [ ]:
# Tool의 필수조건: 데코레이터, Return값과 Tool 설명
# 어떻게 해야 LLM에게 작업이 잘 되었음을 알려줄 수 있을까?
# 더 필요한 기능은 뭘까?
import os

# mode= 'w' or 'a'

@tool
def write_markdown(content: str, file_path: str = "report.md", mode:str ='w'):
    """사용자가 요청한 내용을 markdown 형식으로 file_path에 저장하거나, 기존 파일에 추가합니다.
    mode 파라미터를 통해, 기존 파일을 삭제하거나(w) 기존 파일에 이어 쓰기(a) 할 수 있습니다.
    파일의 기본 저장 경로는 'report.md'이며, 별도의 요청이 있는 경우에는 파일명을 변경하세요.
    
    Args:
        file_path(str): 저장할 파일의 경로
        content(str): 파일에 저장하거나 추가할 마크다운 텍스트의 내용
        mode(str): 파일 편집 모드(w:처음부터 생성, a:뒷부분 추가 작성)
    Returns:
        str: 파일 저장 완료 메시지 및 최종 저장 경로

    """
    with open(file_path, mode, encoding='utf-8') as f:
        f.write(content)
    return f"마크다운 컨텐츠를 {file_path}에 저장 완료했습니다!"

write_markdown.invoke({'content':"## Hello Markdown Edit", 'file_path':'test.md', 'mode':'a'})

툴을 잘 구성했으면, 이제 5개의 툴을 모두 연결합니다.

In [ ]:
tools = [tavily_search, fetch_web, python_repl_tool, current_date, write_markdown]

5개의 툴을 연결하여, 주어진 문제에 대한 답변을 마크다운 보고서로 생성하도록 구성합니다.

In [ ]:
question= 'GPT-5, GPT-5-mini, GPT-5-Nano의 차이에 대해 Reddit 반응을 요약해서 마크다운 리포트로 써줘.'
response = react_agent(llm=llm_nonreasoning, question=question, tools= tools)

In [ ]:
question= 'GPT-5, GPT-5-mini, GPT-5-Nano의 차이에 대해 Reddit 반응을 요약해서 마크다운 리포트로 써줘. 검색은 딱 한번만 하고. gpt.md에 저장해.'
response = react_agent(llm=llm, question=question, tools= tools)

In [ ]:
question= '그록 2.5의 오픈 소스 변환 소식을 gpt.md 뒷부분에 추가해줘.'
response = react_agent(llm=llm_nonreasoning, question=question, tools= tools)

이번에는 Python REPL Tool을 사용해서, 마크다운 리포트에 이미지를 추가할 수 있을지 생각해 보고, 입력을 구성하세요.

In [ ]:
question= '2025년 프로야구에서 8월 승률이 제일 낮은 팀은 어디야? 팀별 승률을 찾고, 그래프로 나타내 주세요. 그래프는 repl tool을 사용해 1.png로 저장하고, 해당 파일을 링크 형태로 삽입하여 baseball.md에 저장하세요.'
response = react_agent(llm=llm_nonreasoning, question=question, tools= tools)

In [ ]:
question= '2025년 프로야구 우승팀은 어디야? 팀별 승률을 찾고, 그래프로 나타내 주세요. 그래프는 repl tool을 사용해 1.png로 저장하고, 해당 파일을 링크 형태로 삽입하여 baseball.md에 저장하세요. 그래프를 그리기 전 더블 체크하세요.'
response = react_agent(llm=llm, question=question, tools= tools)

## System Prompt로 커스텀하기   

이전의 코드는 System Message 없이 구성하는 방식이었지만,   
에이전트의 첫 메시지로 시스템 프롬프트를 추가하면 커스터마이징이 가능합니다.

prompt 항을 추가하고, 해당 내용을 시스템 메시지로 맨 위에 추가하도록 수정하세요.

In [ ]:
def react_agent_v2(llm, question , tools = [tavily_search], prompt=''):
    # 툴과 LLM 구성

    tool_list = {x.name: x for x in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = []
    # 메시지 구성
    if prompt:
        messages.append(SystemMessage(prompt))


    messages.append(HumanMessage(content=question))
    print('Query:', question)

    # LLM에 메시지 전달 (분기)
    tool_msg = llm_with_tools.invoke(question)
    messages.append(tool_msg)
    print('## LLM 호출 ##')
    if tool_msg.content:
        print('LLM:', tool_msg.content)

    while tool_msg.tool_calls:
        # 툴 호출이 있을 경우: 툴 실행 후 결과를 전달 (반복)

        for tool_call in tool_msg.tool_calls:
            tool_name = tool_call['name']

            print(f"-- {tool_name} 사용 중 --")
            print(tool_call)


            tool_exec = tool_list[tool_name]

            tool_result = tool_exec.invoke(tool_call)
            messages.append(tool_result)
        tool_msg = llm_with_tools.invoke(messages)
        print('## LLM 호출 ##')
        messages.append(tool_msg)

        if tool_msg.content:
            print('LLM:', tool_msg.content)

    return messages
    # 메시지 전체 보여주기

In [ ]:
# 문제) 검색을 너무 많이 함(검색어도 좀 김)
# 마크다운 리포트 작성의 'w', 'a', 파일명을 누락함

system_prompt='''
당신은 다양한 툴을 사용해 질문에 답변하기 위한 작업을 수행해야 합니다.
툴별로 아래의 규칙을 따르세요:

기본: 모든 답변은 마크다운 리포트로 저장하세요.
검색: 검색 횟수는 4번까지 수행하세요. 검색어는 5단어 이내로 작성하거나, 질문 형식으로 직접 넣어도 됩니다.
마크다운 저장: 'w', 'a' 중 어떤 모드를 써야 하는지 꼭 판단하여 결정하세요.

'''

question = '''
OpenAI의 PaperBench 실험에 대해 설명하고, 해당 실험에서 가장 높은 성능을 냈던 모델은 무엇인지,
혹시 추가 업데이트된 내용이 있는지 알려줘.
pdf 자료는 fetch하지 말고.
시각화 자료를 첨부한 마크다운 리포트로 써줘.
'''
result = react_agent_v2(llm=llm, question=question, tools = tools, prompt = system_prompt)
print(len(result))
# 메시지의 개수--> Turn

In [ ]:
result[-1]

## 토큰 소모 트래킹하기   

위 코드에서는 LLM을 호출할 때마다 `response_medata`에 input과 output token 수가 포함됩니다.   
LLM을 호출할 때마다 이를 합산하여 최종 결과로 Return할 수 있도록 코드를 수정해 주세요!

In [ ]:
# 이 부분에 옮겨서 진행해 주시면 됩니다 :)
def react_agent_v3(llm, question , tools = [tavily_search], prompt=''):

    token_counter = {'input':0, 'output':0}

    tool_list = {x.name: x for x in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = []
    # 메시지 구성
    if prompt:
        messages.append(SystemMessage(prompt))


    messages.append(HumanMessage(content=question))
    print('Query:', question)

    # LLM에 메시지 전달 (분기)
    tool_msg = llm_with_tools.invoke(question)
    messages.append(tool_msg)
    print('## LLM 호출 ##')

    token_counter['input'] += tool_msg.usage_metadata['input_tokens']
    token_counter['output'] += tool_msg.usage_metadata['output_tokens']

    print('# Tokens// ','Input:', tool_msg.usage_metadata['input_tokens'], 'Output:', tool_msg.usage_metadata['output_tokens'])
        



    if tool_msg.content:
        print('LLM:', tool_msg.content)

    while tool_msg.tool_calls:
        # 툴 호출이 있을 경우: 툴 실행 후 결과를 전달 (반복)

        for tool_call in tool_msg.tool_calls:
            tool_name = tool_call['name']

            print(f"-- {tool_name} 사용 중 --")
            print(tool_call)


            tool_exec = tool_list[tool_name]

            tool_result = tool_exec.invoke(tool_call)
            messages.append(tool_result)
        print('## LLM 호출 ##')
        tool_msg = llm_with_tools.invoke(messages)
        token_counter['input'] += tool_msg.usage_metadata['input_tokens']
        token_counter['output'] += tool_msg.usage_metadata['output_tokens']
        print('# Tokens// ','Input:', tool_msg.usage_metadata['input_tokens'], 'Output:', tool_msg.usage_metadata['output_tokens'])

        messages.append(tool_msg)

        if tool_msg.content:
            print('LLM:', tool_msg.content)

    return messages, token_counter
    # 메시지 전체 보여주기

In [ ]:
system_prompt='''
당신은 다양한 툴을 사용해 질문에 답변하기 위한 작업을 수행해야 합니다.
툴별로 아래의 규칙을 따르세요:

기본: 모든 답변은 마크다운 리포트로 저장하세요.
검색: 검색 횟수는 4번까지 수행하세요. 검색어는 5단어 이내로 작성하거나, 질문 형식으로 직접 넣어도 됩니다.
마크다운 저장: 'w', 'a' 중 어떤 모드를 써야 하는지 꼭 판단하여 결정하세요.

'''
question = '''트랜스포머 구조에 대해 설명해줘'''
result, token_counter  = react_agent_v3(llm=llm, question=question, tools = tools, prompt = system_prompt)
print('# Turns', len(result))
print(token_counter)
# 메시지의 개수--> Turn

In [ ]:
system_prompt='''
당신은 다양한 툴을 사용해 질문에 답변하기 위한 작업을 수행해야 합니다.
툴별로 아래의 규칙을 따르세요:

모든 툴을 한 번에 하나씩만 실행하세요.
여러 개의 툴을 한번에 요청하는 것은 허용하지 않습니다.

기본: 모든 답변은 마크다운 리포트로 저장하세요.
검색: 검색 횟수는 4번까지 수행하세요. 검색어는 5단어 이내로 작성하거나, 질문 형식으로 직접 넣어도 됩니다.
마크다운 저장: 'w', 'a' 중 어떤 모드를 써야 하는지 꼭 판단하여 결정하세요.

'''
question = '''이번 달의 엔비디아 주요 뉴스 요약해줘.'''
result, token_counter  = react_agent_v3(llm=llm, question=question, tools = tools, prompt = system_prompt)
print('# Turns', len(result))
print(token_counter)
# 메시지의 개수--> Turn

## 멀티 에이전트로 분업하기   

현재 에이전트 구조는 매번 컨텍스트를 계속 추가하기 때문에 시간과 비용이 많이 발생합니다.    
검색과 시각화, 보고서 에이전트를 구분하면 어떨까요?

In [ ]:
# 엔비디아 최근 주가 변화 리포트를 쓰기

# 1) 엔비디아 최근 주가 변화를 요약해서 설명해줘, 시각화 할 수 있는 통계를 포함해.
# 2) 주어진 내용을 바탕으로 파이썬 시각화를 해줘.
# 3) 해당 내용을 바탕으로 리포트를 써줘.

question = '''엔비디아 최근 주가 변화를 요약해서 설명해줘, 시각화 할 수 있는 통계를 포함해.'''
result, token_counter  = react_agent_v3(llm=llm_nonreasoning, question=question, tools = [tavily_search],)
print('# Turns', len(result))
print(token_counter)
# 메시지의 개수--> Turn

print(result[-1].content)

In [ ]:
question = '''주어진 내용을 바탕으로 파이썬 시각화를 해줘.'''+ '\n\n'+ result[-1].content


result2, token_counter2  = react_agent_v3(llm=llm, question=question, tools = [python_repl_tool], prompt = system_prompt)
print('# Turns', len(result2))
print(token_counter2)

# 메시지의 개수--> Turn

print(result2[-1].content)

In [ ]:
question = f'''해당 내용을 바탕으로 리포트를 써줘.

사용자의 질문: {question}

조사 결과: {result[-1].content}

시각화 결과: {result2[-1].content}
'''

result3, token_counter3  = react_agent_v3(llm=llm, question=question, tools = [python_repl_tool, write_markdown], prompt = system_prompt)
print('# Turns', len(result3))
print(token_counter3)

# 메시지의 개수--> Turn

print(result3[-1].content)

In [ ]:
token_counter, token_counter2, token_counter3

In [ ]:
@tool
def search_agent(query:str):
    """
    검색 특화 에이전트입니다.
    질문 query가 주어지면 query에 대한 조사를 수행하여 그 결과를 요약하고 전달합니다."""
    response = react_agent_v3(llm=llm_nonreasoning, question=query, tools = [tavily_search])
    return response[0][-1].content

@tool
def python_agent(query:str):
    """
    파이썬 특화 에이전트입니다.
    사용자의 query와 통계자료가 주어지면 파이썬 코드를 이용해 시각화하고 그 결과를 전달합니다."""
    response = react_agent_v3(llm=llm_nonreasoning, question=query, tools = [python_repl_tool])
    return response[0][-1].content


question = '2025년 8월 24일 서울, 부산, 대전, 인천의 아침 기온을 조사해서 알려줘. search_agent와 python_agent한테 자연어로 명령하면 도와줄거야.'

result, token_counter = react_agent_v3(llm=llm, question=question, tools = [search_agent, python_agent, write_markdown])

print(result)